# LeetCode 2869 — Minimum Operations to Collect Elements bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-collect-elements/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi thao tác **lấy phần tử cuối** của `nums` bỏ vào "bộ sưu tập"; ta phải thu thập đủ các số `1, 2, …, k` và muốn dùng **ít thao tác nhất**.
2. Nhận ra bài thực chất là: **duyệt mảng từ phải sang trái** và dừng ngay khi đã thấy đủ `k` giá trị khác nhau trong khoảng `1..k`.
3. Đi từ cách **mô phỏng bằng cách thử từng số thao tác** → **duyệt ngược với `set`** → **đếm "còn thiếu" bằng mảng đã thấy** → **vị trí xuất hiện cuối cùng của mỗi số** → **bitmask**.
4. Biết **tính tay**, viết **trace**, và **kiểm tra chéo** các cách trên với dữ liệu ngẫu nhiên có seed (đúng ràng buộc đề).
5. Tránh các lỗi kinh điển: duyệt từ đầu mảng, đếm trùng giá trị, dùng `x < k` thay vì `x <= k`, quên cộng số thao tác của phần tử cuối cùng, sửa trực tiếp `nums` bằng `pop`.

## 2. Đề bài & ví dụ

Cho một mảng `nums` gồm các số nguyên dương và một số nguyên `k`.

Trong **một thao tác**, bạn **xóa phần tử cuối cùng** của mảng rồi **thêm nó vào bộ sưu tập** của mình.

Trả về **số thao tác tối thiểu** để bộ sưu tập chứa **các phần tử `1, 2, …, k`**.

**Ví dụ 1**

- Input: `nums = [3, 1, 5, 4, 2]`, `k = 2`
- Output: `4`
- Giải thích: sau 4 thao tác, bộ sưu tập lần lượt nhận `2, 4, 5, 1` nên chứa `1` và `2`. Ít hơn 4 thao tác thì chưa có `1`.

**Ví dụ 2**

- Input: `nums = [3, 1, 5, 4, 2]`, `k = 5`
- Output: `5`
- Giải thích: cần đủ `1, 2, 3, 4, 5` nên phải lấy hết cả mảng: 5 thao tác.

**Ví dụ 3**

- Input: `nums = [3, 2, 5, 3, 1]`, `k = 3`
- Output: `4`
- Giải thích: sau 4 thao tác, bộ sưu tập nhận `1, 3, 5, 2` nên chứa `1, 2, 3`.

**Constraints**

- `1 <= nums.length <= 50`
- `1 <= nums[i] <= nums.length`
- `1 <= k <= nums.length`
- Dữ liệu được tạo sao cho **luôn có thể** thu thập đủ các phần tử `1, 2, …, k`.

In [1]:
from typing import List
nums = [3, 1, 5, 4, 2]
print("lấy lần lượt từ cuối:", nums[::-1])     # [2, 4, 5, 1, 3]
print("sau 4 thao tác:", nums[::-1][:4])        # [2, 4, 5, 1]  -> có 1 và 2

lấy lần lượt từ cuối: [2, 4, 5, 1, 3]
sau 4 thao tác: [2, 4, 5, 1]


## 3. Kiến thức nền

### 3.1. Lấy phần tử từ cuối danh sách

Chỉ số âm: `nums[-1]` là phần tử cuối, `nums[-2]` kế cuối. `nums[::-1]` là bản **đảo ngược**. `reversed(nums)` là bộ lặp ngược, không tạo bản sao. `nums[-r:]` là `r` phần tử cuối (với `r ≥ 1`).

In [2]:
nums = [3, 1, 5, 4, 2]
print(nums[-1], nums[-2])        # 2 4
print(nums[::-1])                # [2, 4, 5, 1, 3]
print(list(reversed(nums)))      # [2, 4, 5, 1, 3]
print(nums[-3:])                 # [5, 4, 2]  (3 phần tử cuối)

2 4
[2, 4, 5, 1, 3]
[2, 4, 5, 1, 3]
[5, 4, 2]


### 3.2. `list.pop()` — đúng với "thao tác" của đề, nhưng **sửa** mảng gốc

`nums.pop()` bỏ và trả về phần tử cuối. Dùng nó mô phỏng đề rất sát, nhưng nó **thay đổi** danh sách của người gọi. Trong code nộp bài ta thường **không nên** sửa input; duyệt ngược là đủ.

In [3]:
a = [3, 1, 5]
x = a.pop()
print(x, a)          # 5 [3, 1]

5 [3, 1]


### 3.3. `set` — tập hợp các giá trị khác nhau

`set` lưu mỗi giá trị **một lần**, kiểm tra `x in s` nhanh (`O(1)` trung bình). `s.add(x)` thêm. `len(s)` là số giá trị khác nhau. `A <= B` (hay `A.issubset(B)`) hỏi "A có nằm trong B không".

In [4]:
s = set()
for x in [2, 4, 2, 1]:
    s.add(x)
print(s, len(s))                          # {1, 2, 4} 3  (số 2 chỉ tính một lần)
print(2 in s, 3 in s)                     # True False
need = set(range(1, 3))                   # {1, 2}
print(need, need <= s)                    # {1, 2} True

{1, 2, 4} 3
True False
{1, 2} True


### 3.4. `range(1, k + 1)`

`range(1, k + 1)` sinh `1, 2, …, k` (điểm cuối **không** gồm). Đây là các số ta cần thu thập.

In [5]:
print(list(range(1, 3 + 1)))      # [1, 2, 3]
print(set(range(1, 5 + 1)))       # {1, 2, 3, 4, 5}

[1, 2, 3]
{1, 2, 3, 4, 5}


### 3.5. Thoát sớm: `return` / `break` trong vòng lặp

Khi đã thỏa điều kiện, ta `return` ngay mà không duyệt tiếp. `for i, x in enumerate(reversed(nums), start=1)` cho `i = 1, 2, …` là **số thao tác đã dùng** và `x` là phần tử vừa lấy.

In [6]:
nums = [3, 1, 5, 4, 2]
for ops, x in enumerate(reversed(nums), start=1):
    print("thao tác", ops, "lấy được", x)
    if ops == 3:
        break

thao tác 1 lấy được 2
thao tác 2 lấy được 4
thao tác 3 lấy được 5


### 3.5b. Bitmask (dành cho Cách 5)

Dùng một số nguyên như "bảng bật / tắt": bit thứ `v - 1` bằng `1` nghĩa là "đã thấy số `v`". `mask |= 1 << (v - 1)` bật bit; `(1 << k) - 1` là mặt nạ `k` bit `1` (đã thấy đủ `1..k`).

In [7]:
k = 3
full = (1 << k) - 1
print(format(full, "03b"))         # 111
mask = 0
for v in [1, 3]:
    mask |= 1 << (v - 1)
    print("thấy", v, "->", format(mask, "03b"))   # 001 rồi 101
print(mask == full)                # False (còn thiếu số 2)

111
thấy 1 -> 001
thấy 3 -> 101
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Vì chỉ được lấy **từ cuối**, thứ tự ta thấy các phần tử là thứ tự **đảo ngược** của mảng. Sau `r` thao tác, ta đã có đúng `r` phần tử cuối của `nums`. Ta cần tìm `r` **nhỏ nhất** sao cho `r` phần tử cuối chứa đủ `1..k`.

Tính tay ví dụ 3: `nums = [3, 2, 5, 3, 1]`, `k = 3`, cần `{1, 2, 3}`.

| thao tác `r` | phần tử vừa lấy | bộ sưu tập (chỉ giữ số ≤ k) | đủ `{1,2,3}`? |
|---|---|---|---|
| 1 | 1 | {1} | không |
| 2 | 3 | {1, 3} | không |
| 3 | 5 | {1, 3} (5 > k, bỏ qua) | không |
| 4 | 2 | {1, 2, 3} | **có** |

Đáp án `4`. Chú ý: số `5` bị lấy vào nhưng vô dụng; số `3` ở đầu mảng (vị trí 0) không cần vì ta đã có `3` từ vị trí 3.

## 5. Cách 1 — Brute Force / baseline: thử từng số thao tác

Ý tưởng: thử `r = 1, 2, …, n`. Với mỗi `r`, lấy `r` phần tử cuối `nums[n - r:]`, đổi thành `set`, kiểm tra có chứa `{1..k}` không. `r` đầu tiên thỏa chính là đáp án.

In [8]:
def min_ops_bruteforce(nums: List[int], k: int) -> int:
    n = len(nums)
    need = set(range(1, k + 1))
    for r in range(1, n + 1):
        collected = set(nums[n - r:])      # r phần tử cuối
        if need <= collected:
            return r
    return -1        # không xảy ra theo đề

print(min_ops_bruteforce([3, 1, 5, 4, 2], 2))   # 4
print(min_ops_bruteforce([3, 1, 5, 4, 2], 5))   # 5
print(min_ops_bruteforce([3, 2, 5, 3, 1], 3))   # 4

4
5
4


**Độ phức tạp:** với mỗi `r` ta dựng lại `set` từ đầu tốn `O(r)`; tổng `1 + 2 + … + n = O(n²)`. **Time `O(n²)`**, **Space `O(n)`** cho `set`.

**Nút thắt:** mỗi lần tăng `r` ta dựng lại tập từ đầu, trong khi chỉ có **một phần tử mới**. Hãy cập nhật dần.

## 6. Ý tưởng thuật toán / bước đột phá

Khi `r` tăng thêm 1, bộ sưu tập chỉ nhận **thêm đúng một phần tử** `nums[n - r]`. Nên ta:

1. giữ một `set` đang có, mỗi bước chỉ `add` phần tử mới;
2. dừng ngay khi `set` chứa đủ `1..k`.

Quan sát phụ: các số **lớn hơn `k`** hoàn toàn vô dụng; ta chỉ quan tâm số trong `[1, k]`. Nếu chỉ thêm số `≤ k` vào `set` thì "đủ" ⇔ `len(set) == k` — không cần so sánh tập.

Quan sát thứ hai (Cách 4): số `v` được thu thập ở thao tác thứ `n - (chỉ số xuất hiện cuối cùng của v)`. Đáp án là **lớn nhất** trong các số thao tác đó (phải có đủ mọi `v`), tức là `n - min(last[v])`.

## 7. Các cách tiếp theo

### Cách 2 — Duyệt ngược với `set`

**Giải thích từng đoạn.**

- `seen = set()` chứa các giá trị trong `1..k` đã thu thập.
- `for ops, x in enumerate(reversed(nums), start=1)` mô phỏng từng thao tác: `ops` là số thao tác đã dùng, `x` là phần tử vừa lấy.
- `if x <= k: seen.add(x)` chỉ ghi nhận số hữu ích (`x ≥ 1` luôn đúng theo đề).
- `if len(seen) == k: return ops` đủ `k` giá trị khác nhau trong `[1, k]` thì dừng.

In [9]:
def min_ops_set(nums: List[int], k: int) -> int:
    seen = set()
    for ops, x in enumerate(reversed(nums), start=1):
        if x <= k:
            seen.add(x)
        if len(seen) == k:
            return ops
    return -1

print(min_ops_set([3, 1, 5, 4, 2], 2))   # 4
print(min_ops_set([3, 2, 5, 3, 1], 3))   # 4

4
4


In [10]:
# Code nộp LeetCode cho Cách 2
class SolutionSet:
    def minOperations(self, nums: List[int], k: int) -> int:
        seen = set()
        for ops, x in enumerate(reversed(nums), start=1):
            if x <= k:
                seen.add(x)
            if len(seen) == k:
                return ops
        return -1

print(SolutionSet().minOperations([3, 1, 5, 4, 2], 5))   # 5

5


**Độ phức tạp Cách 2:** duyệt tối đa `n` phần tử, mỗi bước `add` `O(1)` trung bình. **Time `O(n)`**, **Space `O(k)`** (`set` chứa tối đa `k` giá trị).

---

### Cách 3 — Đếm "còn thiếu" bằng mảng đã thấy

**Giải thích.** Thay `set` bằng mảng `seen` kích thước `k + 1` kiểu `bool` và biến `missing = k` (còn thiếu bao nhiêu số). Gặp `x ≤ k` mà chưa thấy → đánh dấu và `missing -= 1`; khi `missing == 0` thì dừng. Cách này không dùng cấu trúc băm, hợp khi giá trị là chỉ số nhỏ.

In [11]:
def min_ops_array(nums: List[int], k: int) -> int:
    seen = [False] * (k + 1)       # seen[v] = đã thu thập số v chưa (bỏ ô 0)
    missing = k
    for ops, x in enumerate(reversed(nums), start=1):
        if x <= k and not seen[x]:
            seen[x] = True
            missing -= 1
            if missing == 0:
                return ops
    return -1

print(min_ops_array([3, 1, 5, 4, 2], 2))   # 4
print(min_ops_array([3, 2, 5, 3, 1], 3))   # 4

4
4


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionArray:
    def minOperations(self, nums: List[int], k: int) -> int:
        seen = [False] * (k + 1)
        missing = k
        ops = 0
        for x in reversed(nums):
            ops += 1
            if x <= k and not seen[x]:
                seen[x] = True
                missing -= 1
                if missing == 0:
                    return ops
        return ops

print(SolutionArray().minOperations([3, 2, 5, 3, 1], 3))   # 4

4


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(k)`**.

---

### Cách 4 — Vị trí xuất hiện **cuối cùng** của mỗi số

**Giải thích.** Số `v` được lấy ra sớm nhất khi ta gặp **lần xuất hiện cuối cùng** của nó trong mảng (vì ta đi từ cuối). Gọi `last[v]` là chỉ số lớn nhất mà `nums[i] == v`. Khi đã lấy các phần tử từ chỉ số `p` tới cuối, ta có đủ `1..k` ⇔ `p ≤ last[v]` với mọi `v ∈ [1, k]` ⇔ `p ≤ min(last[v])`. Cần `n - p` thao tác nhỏ nhất, nên chọn `p = min(last[v])`. Đáp án `n - min(last[v] for v in 1..k)`.

Ghi `last` bằng dict: duyệt xuôi, `last[x] = i` (lần ghi sau ghi đè lần trước).

In [13]:
def min_ops_last(nums: List[int], k: int) -> int:
    last = {}
    for i, x in enumerate(nums):
        last[x] = i                      # lần sau ghi đè -> giữ chỉ số cuối cùng
    return len(nums) - min(last[v] for v in range(1, k + 1))

print(min_ops_last([3, 1, 5, 4, 2], 2))   # 5 - min(last[1]=1, last[2]=4) = 4
print(min_ops_last([3, 1, 5, 4, 2], 5))   # 5 - 0 = 5
print(min_ops_last([3, 2, 5, 3, 1], 3))   # 5 - min(4, 1, 3) = 4

4
5
4


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionLast:
    def minOperations(self, nums: List[int], k: int) -> int:
        last = {}
        for i, x in enumerate(nums):
            last[x] = i
        return len(nums) - min(last[v] for v in range(1, k + 1))

print(SolutionLast().minOperations([3, 1, 5, 4, 2], 2))   # 4

4


**Độ phức tạp Cách 4:** **Time `O(n + k)`** = `O(n)`, **Space `O(n)`** cho dict `last`. Cách này dựa vào đảm bảo của đề rằng mọi `v ∈ [1, k]` đều xuất hiện (nếu không `last[v]` gây `KeyError`).

---

### Cách 5 — Bitmask

**Giải thích.** `mask` là một số nguyên mà bit `v - 1` bằng `1` nếu đã thu thập số `v`. `full = (1 << k) - 1`. Duyệt ngược; gặp `x ≤ k` thì `mask |= 1 << (x - 1)`; khi `mask == full` trả về `ops`.

In [15]:
def min_ops_mask(nums: List[int], k: int) -> int:
    full = (1 << k) - 1
    mask = 0
    for ops, x in enumerate(reversed(nums), start=1):
        if x <= k:
            mask |= 1 << (x - 1)
        if mask == full:
            return ops
    return -1

print(min_ops_mask([3, 1, 5, 4, 2], 2))   # 4
print(min_ops_mask([3, 2, 5, 3, 1], 3))   # 4

4
4


In [16]:
# Code nộp LeetCode cho Cách 5
class SolutionMask:
    def minOperations(self, nums: List[int], k: int) -> int:
        full = (1 << k) - 1
        mask = 0
        for ops, x in enumerate(reversed(nums), start=1):
            if x <= k:
                mask |= 1 << (x - 1)
            if mask == full:
                return ops
        return len(nums)

print(SolutionMask().minOperations([3, 1, 5, 4, 2], 5))   # 5

5


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** (một số nguyên `k ≤ 50` bit).

## 8. TRACE từng bước

Trace Cách 2 (duyệt ngược + `set`) trên `nums = [3, 2, 5, 3, 1]`, `k = 3`:

| ops | `x` lấy ra | `x <= k`? | `seen` sau | `len(seen)` | dừng? |
|---|---|---|---|---|---|
| 1 | 1 | có | {1} | 1 | không |
| 2 | 3 | có | {1, 3} | 2 | không |
| 3 | 5 | không | {1, 3} | 2 | không |
| 4 | 2 | có | {1, 2, 3} | 3 | **có, trả 4** |

In [17]:
def trace_set(nums, k):
    seen = set()
    print(f"nums={nums}, k={k}")
    for ops, x in enumerate(reversed(nums), start=1):
        useful = x <= k
        if useful:
            seen.add(x)
        print(f"ops={ops}: lấy {x}, hữu ích={useful}, seen={sorted(seen)}, len={len(seen)}")
        if len(seen) == k:
            print("-> đủ", k, "số, dừng. Đáp án =", ops)
            return ops
    return -1

trace_set([3, 2, 5, 3, 1], 3)
trace_set([3, 1, 5, 4, 2], 2)

nums=[3, 2, 5, 3, 1], k=3
ops=1: lấy 1, hữu ích=True, seen=[1], len=1
ops=2: lấy 3, hữu ích=True, seen=[1, 3], len=2
ops=3: lấy 5, hữu ích=False, seen=[1, 3], len=2
ops=4: lấy 2, hữu ích=True, seen=[1, 2, 3], len=3
-> đủ 3 số, dừng. Đáp án = 4
nums=[3, 1, 5, 4, 2], k=2
ops=1: lấy 2, hữu ích=True, seen=[2], len=1
ops=2: lấy 4, hữu ích=False, seen=[2], len=1
ops=3: lấy 5, hữu ích=False, seen=[2], len=1
ops=4: lấy 1, hữu ích=True, seen=[1, 2], len=2
-> đủ 2 số, dừng. Đáp án = 4


4

In [18]:
def trace_last(nums, k):
    last = {}
    for i, x in enumerate(nums):
        last[x] = i
    print("chỉ số cuối cùng của từng giá trị:", dict(sorted(last.items())))
    needed = {v: last[v] for v in range(1, k + 1)}
    print("chỉ số cuối của 1..k:", needed)
    p = min(needed.values())
    print(f"p = min = {p} -> số thao tác = n - p = {len(nums)} - {p} = {len(nums) - p}")
    return len(nums) - p

trace_last([3, 2, 5, 3, 1], 3)

chỉ số cuối cùng của từng giá trị: {1: 4, 2: 1, 3: 3, 5: 2}
chỉ số cuối của 1..k: {1: 4, 2: 1, 3: 3}
p = min = 1 -> số thao tác = n - p = 5 - 1 = 4


4

## 9. Code nộp LeetCode

In [19]:
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        seen = set()
        for ops, x in enumerate(reversed(nums), start=1):
            if x <= k:
                seen.add(x)
            if len(seen) == k:
                return ops
        return len(nums)

print(Solution().minOperations([3, 1, 5, 4, 2], 2))   # 4
print(Solution().minOperations([3, 1, 5, 4, 2], 5))   # 5
print(Solution().minOperations([3, 2, 5, 3, 1], 3))   # 4

4
5
4


## 10. Vì sao thuật toán đúng

- **Mô hình đúng:** sau `r` thao tác ta có đúng `r` phần tử cuối của `nums`, theo thứ tự đảo ngược. Duyệt `reversed(nums)` mô phỏng chính xác.
- **Invariant:** sau thao tác thứ `ops`, `seen` là tập các giá trị thuộc `[1, k]` nằm trong `ops` phần tử cuối. Mỗi bước thêm đúng phần tử mới nên invariant giữ nguyên.
- **Điều kiện dừng:** vì `seen ⊆ {1, …, k}`, `len(seen) == k` ⇔ `seen = {1, …, k}` ⇔ đã thu thập đủ.
- **Tối thiểu:** ta trả về ngay ở `ops` đầu tiên thỏa điều kiện, mà điều kiện chỉ có thể trở thành đúng khi `ops` tăng (tập thu thập chỉ lớn lên, không bao giờ mất phần tử), nên không có `ops` nhỏ hơn thỏa mãn.
- **Luôn có đáp án:** đề đảm bảo `1..k` đều xuất hiện, nên tới `ops = n` (lấy hết mảng) chắc chắn đủ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `k ≤ n`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng `r`, dựng lại set | `O(n²)` | `O(n)` |
| 2. Duyệt ngược + set | `O(n)` | `O(k)` |
| 3. Mảng đã thấy + đếm thiếu | `O(n)` | `O(k)` |
| 4. Vị trí cuối của mỗi số | `O(n)` | `O(n)` |
| 5. Bitmask | `O(n)` | `O(1)` |

Với `n ≤ 50` mọi cách đều rất nhanh; điểm cần học là cách **cập nhật dần** thay vì dựng lại từ đầu.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [20]:
sols = [SolutionSet(), SolutionArray(), SolutionLast(), SolutionMask(), Solution()]

cases = [
    ("ví dụ 1", [3, 1, 5, 4, 2], 2, 4),
    ("ví dụ 2", [3, 1, 5, 4, 2], 5, 5),
    ("ví dụ 3", [3, 2, 5, 3, 1], 3, 4),
    ("một phần tử", [1], 1, 1),
    ("k=1 số 1 ở cuối", [2, 2, 1], 1, 1),
    ("k=1 số 1 ở đầu", [1, 2, 2], 1, 3),
    ("số cần ở rất xa", [1, 2, 3, 4, 5, 5, 5, 5], 3, 8),
    ("đảo ngược hoàn hảo", [3, 2, 1], 3, 3),
    ("trùng lặp nhiều", [2, 1, 2, 1, 2, 1], 2, 2),
]
for name, nums, k, exp in cases:
    assert min_ops_bruteforce(nums, k) == exp, (name, "brute", min_ops_bruteforce(nums, k), exp)
    for s in sols:
        got = s.minOperations(list(nums), k)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 4
PASS ví dụ 2 -> 5
PASS ví dụ 3 -> 4
PASS một phần tử -> 1
PASS k=1 số 1 ở cuối -> 1
PASS k=1 số 1 ở đầu -> 3
PASS số cần ở rất xa -> 8
PASS đảo ngược hoàn hảo -> 3
PASS trùng lặp nhiều -> 2


### 12.2. Kiểm tra không sửa input và phản ví dụ

Code nộp phải **không thay đổi** `nums` của người gọi. Phản ví dụ: mảng có số lớn hơn `k` (vô dụng) và số trùng.

In [21]:
nums = [3, 1, 5, 4, 2]
copy = nums[:]
for s in sols:
    s.minOperations(nums, 2)
assert nums == copy
print("input không bị sửa")

# số trùng không được đếm hai lần: cần {1,2,3}; từ cuối: 1,1,1,2,3 -> đủ ở thao tác 5
assert Solution().minOperations([3, 2, 1, 1, 1], 3) == 5
# số > k vô dụng: cần {1,2}; cuối: 9,8,7,2,1 -> 5 thao tác
assert Solution().minOperations([1, 2, 7, 8, 9], 2) == 5
print("PASS phản ví dụ trùng / số lớn hơn k")

input không bị sửa
PASS phản ví dụ trùng / số lớn hơn k


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `n ∈ [1, 50]`, `k ∈ [1, n]`, `nums[i] ∈ [1, n]`. Để đảm bảo `1..k` đều xuất hiện, chọn ngẫu nhiên `k` vị trí rồi đặt `1..k` (xáo trộn) vào đó.

In [22]:
import random
random.seed(2869)

def gen():
    n = random.randint(1, 50)
    k = random.randint(1, n)
    nums = [random.randint(1, n) for _ in range(n)]
    pos = random.sample(range(n), k)
    vals = list(range(1, k + 1))
    random.shuffle(vals)
    for p, v in zip(pos, vals):
        nums[p] = v
    return nums, k

for t in range(1000):
    nums, k = gen()
    assert set(range(1, k + 1)) <= set(nums)           # đúng đảm bảo của đề
    exp = min_ops_bruteforce(nums, k)
    for s in sols:
        assert s.minOperations(list(nums), k) == exp, (t, type(s).__name__, nums, k)
    assert 1 <= exp <= len(nums)
print("1000 test ngẫu nhiên: mọi cách khớp brute force")

1000 test ngẫu nhiên: mọi cách khớp brute force


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Duyệt từ **đầu** mảng

In [23]:
def sai_duyet_xuoi(nums, k):
    seen = set()
    for ops, x in enumerate(nums, start=1):      # sai: phải đi từ cuối
        if x <= k:
            seen.add(x)
        if len(seen) == k:
            return ops

print("sai :", sai_duyet_xuoi([3, 1, 5, 4, 2], 2))      # 2
print("đúng:", Solution().minOperations([3, 1, 5, 4, 2], 2))   # 4

sai : 5
đúng: 4


**Vì sao sai:** đề chỉ cho lấy **phần tử cuối**; thứ tự thu thập là từ phải sang trái.

### Lỗi 2: Đếm số lượng phần tử `≤ k` thay vì số giá trị **khác nhau**

In [24]:
def sai_dem_trung(nums, k):
    cnt = 0
    for ops, x in enumerate(reversed(nums), start=1):
        if x <= k:
            cnt += 1                  # sai: số trùng bị đếm nhiều lần
        if cnt == k:
            return ops

print("sai :", sai_dem_trung([3, 2, 1, 1, 1], 3))       # 3 (sai: chưa có đủ 1,2,3)
print("đúng:", Solution().minOperations([3, 2, 1, 1, 1], 3))   # 5

sai : 3
đúng: 5


**Vì sao sai:** lấy ba số `1` liên tiếp không giúp có `2` hay `3`. Phải đếm **giá trị khác nhau** (dùng `set` / mảng đã thấy).

### Lỗi 3: `x < k` thay vì `x <= k`

In [25]:
def sai_nho_hon(nums, k):
    seen = set()
    for ops, x in enumerate(reversed(nums), start=1):
        if x < k:                    # sai: bỏ sót chính số k
            seen.add(x)
        if len(seen) == k:
            return ops
    return None

print("sai :", sai_nho_hon([3, 1, 5, 4, 2], 2))         # None (không bao giờ đủ)
print("đúng:", Solution().minOperations([3, 1, 5, 4, 2], 2))   # 4

sai : None
đúng: 4


**Vì sao sai:** ta cần cả `1, …, k` **gồm `k`**. Điều kiện đúng là `x <= k`.

### Lỗi 4: Dùng `pop()` làm hỏng mảng của người gọi

In [26]:
def sai_pop(nums, k):
    seen = set()
    ops = 0
    while len(seen) < k:
        x = nums.pop()               # sai: sửa nums của người gọi
        ops += 1
        if x <= k:
            seen.add(x)
    return ops

data = [3, 1, 5, 4, 2]
print("trả về:", sai_pop(data, 2), "| mảng sau khi gọi:", data)     # mảng bị mất phần tử
data = [3, 1, 5, 4, 2]
print("đúng :", Solution().minOperations(data, 2), "| mảng sau khi gọi:", data)

trả về: 4 | mảng sau khi gọi: [3]
đúng : 4 | mảng sau khi gọi: [3, 1, 5, 4, 2]


**Vì sao sai:** đáp án đúng nhưng **phá hủy input**. Trong dự án thật (và khi test nhiều lần trên cùng mảng) đây là nguồn lỗi khó tìm. Duyệt `reversed(nums)` không sửa gì.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chỉ được lấy / xóa **từ một đầu**" → thứ tự xử lý đã cố định, hãy **duyệt từ đầu đó** thay vì mô phỏng xóa.
- "Cần **đủ tập** `1..k`" → `set` + so `len(...) == k`, hoặc bitmask, hoặc biến "còn thiếu".
- "**Ít thao tác nhất**" với thao tác chỉ làm tập lớn dần → **dừng ngay lần đầu thỏa** (không cần tối ưu thêm).
- Giá trị trong `[1, n]` → có thể dùng **mảng bool** thay `set`; `k` nhỏ → **bitmask**.
- Cách "vị trí cuối cùng" `n - min(last[v])` gọn nhưng phụ thuộc đảm bảo "mọi `v` đều có mặt".

## 15. Biến thể của bài

1. **Lấy từ đầu hoặc cuối** tùy chọn (tối thiểu hóa tổng thao tác): cần hai con trỏ / duyệt các cách chia (liên quan LeetCode 1423 *Maximum Points You Can Obtain from Cards*).
2. **Không đảm bảo có đủ `1..k`:** trả `-1` nếu không thể.
3. **Tập cần thu thập là một tập bất kỳ** `T` (không phải `1..k`): dùng `set(T)` và "còn thiếu".
4. **Đếm số lần cần thu thập mỗi giá trị đúng `c` lần** (dùng `Counter`).
5. **Cửa sổ ngắn nhất chứa đủ tập** (một đầu cố định ở cuối): gần với LeetCode 76 *Minimum Window Substring*.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng `r` | dựng set từ `r` phần tử cuối | `O(n²)` | `O(n)` | chắc chắn đúng, để đối chiếu |
| 2. Duyệt ngược + set | thêm dần, dừng khi `len == k` | `O(n)` | `O(k)` | **cách nộp bài** |
| 3. Mảng bool + đếm thiếu | đánh dấu đã thấy | `O(n)` | `O(k)` | giá trị là chỉ số nhỏ |
| 4. Vị trí cuối | `n - min(last[v])` | `O(n)` | `O(n)` | ngắn, cần đảm bảo `1..k` có mặt |
| 5. Bitmask | mỗi bit là một số đã thấy | `O(n)` | `O(1)` | `k` nhỏ, luyện bit |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(reversed([4, 7, 9]))`, `set([2, 2, 5]) == {2, 5}`, `{1, 2} <= {1, 2, 3}`, `(1 << 3) - 1`.

**Bài 2 (tính tay).** Với `nums = [2, 4, 1, 3, 3, 5]`, `k = 3`: viết dãy phần tử lấy ra từng thao tác, `seen` sau mỗi bước và số thao tác tối thiểu.

**Bài 3 (điền TODO).** Hoàn thành `min_ops_todo(nums, k)` bằng duyệt ngược + set.

In [27]:
def min_ops_todo(nums, k):
    seen = set()
    # TODO 1: for ops, x in enumerate(reversed(nums), start=1):
    # TODO 2:     nếu x <= k thì seen.add(x)
    # TODO 3:     nếu len(seen) == k thì return ops
    return len(nums)

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `min_ops_last(nums, k)` (Cách 4, vị trí cuối cùng) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_ops_or_minus1(nums, k)`: trả `-1` nếu không thể thu thập đủ `1..k` (đề không còn đảm bảo). (b) Viết `min_ops_from_either_end(nums, k)`: mỗi thao tác được lấy phần tử **đầu hoặc cuối**; trả về số thao tác tối thiểu (gợi ý: thử mọi cặp `(a, b)` lấy `a` phần tử đầu và `b` phần tử cuối).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `list(reversed([4, 7, 9])) = [9, 7, 4]`; `set([2, 2, 5]) == {2, 5}` là `True`; `{1, 2} <= {1, 2, 3}` là `True`; `(1 << 3) - 1 = 7`. Bài 2: đảo mảng được `5, 3, 3, 1, 4, 2`.
- **Mức 2 (vừa):** Bài 3: ba dòng đã ghi trong comment. Bài 4: duyệt xuôi lưu `last[x] = i`, rồi `n - min(...)`.
- **Mức 3 (sâu):** Bài 5(a): nếu cuối vòng `len(seen) < k` thì trả `-1`. Bài 5(b): với mỗi `a` từ `0..n` và `b` từ `0..n-a`, tập thu thập là `nums[:a] + nums[n-b:]` (đầu `a` phần tử, cuối `b` phần tử); chọn `a + b` nhỏ nhất thỏa đủ `1..k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert list(reversed([4, 7, 9])) == [9, 7, 4]
assert set([2, 2, 5]) == {2, 5}
assert {1, 2} <= {1, 2, 3}
assert (1 << 3) - 1 == 7

# Bài 2: nums = [2,4,1,3,3,5], k = 3. Lấy: 5 (seen {}), 3 ({3}), 3 ({3}), 1 ({1,3}), 4 ({1,3}), 2 ({1,2,3}) -> 6 thao tác
assert Solution().minOperations([2, 4, 1, 3, 3, 5], 3) == 6

# Bài 3
def min_ops_todo_ans(nums, k):
    seen = set()
    for ops, x in enumerate(reversed(nums), start=1):
        if x <= k:
            seen.add(x)
        if len(seen) == k:
            return ops
    return len(nums)

assert min_ops_todo_ans([3, 1, 5, 4, 2], 2) == 4
assert min_ops_todo_ans([3, 2, 5, 3, 1], 3) == 4

# Bài 4
def min_ops_last_ans(nums, k):
    last = {}
    for i, x in enumerate(nums):
        last[x] = i
    return len(nums) - min(last[v] for v in range(1, k + 1))

assert min_ops_last_ans([3, 1, 5, 4, 2], 2) == 4
assert min_ops_last_ans([3, 1, 5, 4, 2], 5) == 5
assert min_ops_last_ans([3, 2, 5, 3, 1], 3) == 4

# Bài 5(a)
def min_ops_or_minus1(nums, k):
    seen = set()
    for ops, x in enumerate(reversed(nums), start=1):
        if x <= k:
            seen.add(x)
        if len(seen) == k:
            return ops
    return -1

assert min_ops_or_minus1([1, 1, 1], 2) == -1
assert min_ops_or_minus1([3, 1, 5, 4, 2], 2) == 4

# Bài 5(b)
def min_ops_from_either_end(nums, k):
    n = len(nums)
    need = set(range(1, k + 1))
    best = None
    for a in range(n + 1):
        for b in range(n - a + 1):
            got = set(nums[:a]) | set(nums[n - b:] if b else [])
            if need <= got and (best is None or a + b < best):
                best = a + b
    return best

# [3,1,5,4,2], k=2: lấy 2 phần tử đầu [3,1] (có số 1) và 1 phần tử cuối [2] (có số 2) -> 3 thao tác
assert min_ops_from_either_end([3, 1, 5, 4, 2], 2) == 3
# lấy từ hai đầu không bao giờ tệ hơn chỉ lấy từ cuối
assert min_ops_from_either_end([3, 1, 5, 4, 2], 5) == 5
assert min_ops_from_either_end([1, 9, 9, 9, 2], 2) == 2
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (thu thập đủ tập từ một đầu: duyệt từ đầu đó, thêm dần, thoát sớm)

```python
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        seen = set()
        for ops, x in enumerate(reversed(nums), start=1):
            if x <= k:
                seen.add(x)
            if len(seen) == k:
                return ops
        return len(nums)
```

**Checklist**

- [ ] Duyệt từ **cuối** mảng (đúng thứ tự lấy phần tử)?
- [ ] Chỉ ghi nhận giá trị `<= k` và đếm **giá trị khác nhau**?
- [ ] `ops` bắt đầu từ `1` (đếm cả phần tử vừa lấy)?
- [ ] Không sửa `nums` của người gọi?
- [ ] Đã thử `k = 1`, `k = n`, mảng một phần tử?
- [ ] Signature đúng `minOperations(self, nums: List[int], k: int) -> int`?